## 1. Mount Google Drive (Colab) / Check Local Path

In [2]:
# Mount Google Drive (only needed for Colab mode)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted!")
    print("📂 Contents:")
    !ls /content/drive/MyDrive/
except ImportError:
    print("💻 Running locally - Google Drive mount skipped")
    print("   Make sure your video files are in the local path configured below")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted!
📂 Contents:
'~.'
 ♥️
'$S_1 \sim U(1.0s, 2.5s)$'$'\n\n\n''I GET THESE CHARACTERS....gdoc'
'~. (1)'
 11.mov
'1. Introduction'$'\n''1.1 Overview'$'\n''This report covers a....gdoc'
 727821TUIT027.pdf
'727821TUIT027 - Resume (1).pdf'
'727821TUIT027 - Resume (2).pdf'
'727821TUIT027 - Resume.pdf'
 adobe-lightroom-icon.svg
 ai_emotion_project
'Bonafied .pdf'
'Certificates & Reports'
'ChatGPT Image Aug 13, 2025, 01_34_25 AM.png'
'CockroachDB Paper Presentation.gslides'
'Colab Notebooks'
'Copy of Laboratory Work №2: Distributed Request Processing System.gdoc'
'Copy of route Dainava I (1).gsheet'
'Copy of route Dainava I.gsheet'
'Copy of route Dainava I.xlsx'
'Copy of route Kalnieciai.gsheet'
'Copy of route Kalnieciai.xlsx'
 Dataset
'Distributed Algorithms in Cloud Computing:A Review (1).gslides'
'Distributed Algorithms in Cloud Computing:A Revi

## 2. Install Dependencies

**Note:** This cell installs the required AI/ML libraries.

In [3]:
# Install required packages
print("📦 Installing packages (this may take 2-3 minutes)...")

# Check if running on Colab or Local
try:
    import google.colab
    IS_COLAB = True
    print("🌐 Running on Google Colab")
except ImportError:
    IS_COLAB = False
    print("💻 Running on Local Machine")

# Install packages based on environment
if IS_COLAB:
    # For Colab: Try GPU first, fallback to CPU
    !pip uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null
    !pip install -q insightface deepface opencv-python-headless tensorflow

    # Try to install GPU version, fallback to CPU
    import subprocess
    result = subprocess.run(['pip', 'install', '-q', 'onnxruntime-gpu'], capture_output=True)
    if result.returncode != 0:
        print("⚠️ GPU onnxruntime not available, installing CPU version...")
        !pip install -q onnxruntime
else:
    # For Local: Use CPU version (safer)
    !pip install -q insightface onnxruntime deepface opencv-python-headless tensorflow

print("✅ All packages installed!")
print("🔍 Checking hardware support...")

import onnxruntime as ort
print(f"   ONNX Runtime version: {ort.__version__}")
print(f"   Available providers: {ort.get_available_providers()}")

if "CUDAExecutionProvider" in ort.get_available_providers():
    print("   ✅ GPU support available!")
else:
    print("   ℹ️ CPU mode will be used (GPU not available)")

📦 Installing packages (this may take 2-3 minutes)...
🌐 Running on Google Colab
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 7.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.1/133.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.3 MB/s eta 0:00:00
✅ All packages installed!
🔍 Checking hardware support...
   ONNX Runtime vers

## 3. Import Libraries

In [4]:
import os
import json
import time
import cv2
import numpy as np
from pathlib import Path
from datetime import datetime

# DeepFace for face detection AND emotion detection
from deepface import DeepFace

print("✅ Libraries imported successfully")

26-01-02 20:41:09 - Directory /root/.deepface has been created
26-01-02 20:41:09 - Directory /root/.deepface/weights has been created
✅ Libraries imported successfully


## 4. Configuration

**MODIFY THIS CELL:** Set the job_id from your web application.

In [ ]:
# ========================================
# CONFIGURATION - MODIFY THIS!
# ========================================

# ============== TOGGLE: LOCAL vs COLAB ==============
# Set to True to use LOCAL machine, False for Google Colab
USE_LOCAL_MACHINE = True  # ← CHANGE THIS: True for local, False for Colab

# 🔴 IMPORTANT: Get job_id from your web application (http://localhost:3000)
# 1. Upload a video on the web app
# 2. Copy the job_id from the response
# 3. Paste it below, replacing "YOUR_JOB_ID_HERE"

JOB_ID = "186cc8fb-870c-418b-ba3a-c8042ca83a82"  # ← CHANGE THIS TO YOUR JOB ID!

# ============== PATH CONFIGURATION ==============
if USE_LOCAL_MACHINE:
    # LOCAL MACHINE PATHS - Modify if needed
    BASE_PATH = r"C:\Users\Hariharan A\Music\Final lab\backend\media\mock_drive"
    print("💻 Mode: LOCAL MACHINE")
else:
    # GOOGLE COLAB PATHS (Google Drive)
    BASE_PATH = "/content/drive/MyDrive/ai_emotion_project"
    print("🌐 Mode: GOOGLE COLAB")

# Paths (auto-configured based on mode)
INPUT_VIDEO_PATH = f"{BASE_PATH}/input_videos/{JOB_ID}.mp4"
OUTPUT_VIDEO_PATH = f"{BASE_PATH}/output_videos/{JOB_ID}.mp4"
OUTPUT_JSON_PATH = f"{BASE_PATH}/output_json/{JOB_ID}.json"
STATUS_FILE_PATH = f"{BASE_PATH}/status/{JOB_ID}.json"

# Ensure folders exist
import os
os.makedirs(f"{BASE_PATH}/input_videos", exist_ok=True)
os.makedirs(f"{BASE_PATH}/output_videos", exist_ok=True)
os.makedirs(f"{BASE_PATH}/output_json", exist_ok=True)
os.makedirs(f"{BASE_PATH}/status", exist_ok=True)

print(f"\n✅ Configuration set for job: {JOB_ID}")
print(f"📹 Input video: {INPUT_VIDEO_PATH}")
print(f"📹 Output video: {OUTPUT_VIDEO_PATH}")
print(f"📊 Output JSON: {OUTPUT_JSON_PATH}")
print(f"📝 Status file: {STATUS_FILE_PATH}")

# Verify input video exists
if os.path.exists(INPUT_VIDEO_PATH):
    size_mb = os.path.getsize(INPUT_VIDEO_PATH) / (1024 * 1024)
    print(f"✅ Input video found! Size: {size_mb:.2f} MB")
else:
    print(f"⚠️ Input video NOT found at: {INPUT_VIDEO_PATH}")
    print("   Make sure you've uploaded the video first!")

🌐 Mode: GOOGLE COLAB

✅ Configuration set for job: 186cc8fb-870c-418b-ba3a-c8042ca83a82
📹 Input video: /content/drive/MyDrive/ai_emotion_project/input_videos/186cc8fb-870c-418b-ba3a-c8042ca83a82.mp4
📹 Output video: /content/drive/MyDrive/ai_emotion_project/output_videos/186cc8fb-870c-418b-ba3a-c8042ca83a82.mp4
📊 Output JSON: /content/drive/MyDrive/ai_emotion_project/output_json/186cc8fb-870c-418b-ba3a-c8042ca83a82.json
📝 Status file: /content/drive/MyDrive/ai_emotion_project/status/186cc8fb-870c-418b-ba3a-c8042ca83a82.json
✅ Input video found! Size: 0.25 MB


## 5. Helper Functions

In [6]:
# ========================================
# HELPER FUNCTIONS
# ========================================

import numpy as np
from collections import defaultdict

def update_status(status, message, progress=None):
    """Update the status file for the web application to read."""
    status_data = {
        "status": status,
        "message": message,
        "timestamp": time.time()
    }
    if progress is not None:
        status_data["progress"] = progress

    with open(STATUS_FILE_PATH, 'w') as f:
        json.dump(status_data, f, indent=2)
    print(f"📝 Status: {status} - {message}")


def check_termination():
    """Check if the job has been terminated by the user."""
    try:
        if os.path.exists(STATUS_FILE_PATH):
            with open(STATUS_FILE_PATH, 'r') as f:
                status_data = json.load(f)
                if status_data.get("status") == "CANCELLED" or status_data.get("terminate") == True:
                    print("🛑 Job termination detected!")
                    return True
    except Exception as e:
        pass
    return False


class FaceTracker:
    """Track faces across frames using embedding similarity."""
    def __init__(self, similarity_threshold=0.65, max_disappeared=30, update_alpha=0.9):
        self.next_person_id = 1
        self.active_people = {}
        self.similarity_threshold = float(similarity_threshold)
        self.max_disappeared = int(max_disappeared)
        self.update_alpha = float(update_alpha)
        self.disappeared_frames = defaultdict(int)

    def _normalize(self, emb):
        emb = np.asarray(emb, dtype=np.float32).reshape(-1)
        norm = float(np.linalg.norm(emb) + 1e-12)
        return emb / norm

    def get_cosine_similarity(self, embedding1, embedding2):
        e1 = self._normalize(embedding1)
        e2 = self._normalize(embedding2)
        return float(np.dot(e1, e2))

    def _register(self, embedding, timestamp):
        person_id = self.next_person_id
        self.active_people[person_id] = {
            "embedding": self._normalize(embedding),
            "last_seen": float(timestamp),
        }
        self.disappeared_frames[person_id] = 0
        self.next_person_id += 1
        return person_id

    def _mark_disappeared(self):
        for person_id in list(self.active_people.keys()):
            self.disappeared_frames[person_id] += 1
            if self.disappeared_frames[person_id] > self.max_disappeared:
                del self.active_people[person_id]
                del self.disappeared_frames[person_id]

    def update(self, faces, timestamp):
        if not faces:
            self._mark_disappeared()
            return []

        if not self.active_people:
            assigned = []
            for face_data in faces:
                assigned.append(self._register(face_data["embedding"], timestamp))
            return assigned

        person_ids = list(self.active_people.keys())
        stored = np.stack([self.active_people[pid]["embedding"] for pid in person_ids], axis=0)

        new_embeddings = [self._normalize(f["embedding"]) for f in faces]
        new_mat = np.stack(new_embeddings, axis=0)

        sim = new_mat @ stored.T

        assigned_ids = [None] * len(faces)
        used_people = set()
        used_faces = set()

        candidates = []
        for i in range(sim.shape[0]):
            for j in range(sim.shape[1]):
                candidates.append((float(sim[i, j]), i, j))
        candidates.sort(reverse=True, key=lambda x: x[0])

        for s, i, j in candidates:
            if s < self.similarity_threshold:
                break
            if i in used_faces:
                continue
            pid = person_ids[j]
            if pid in used_people:
                continue

            assigned_ids[i] = pid
            used_faces.add(i)
            used_people.add(pid)

            old_emb = self.active_people[pid]["embedding"]
            new_emb = new_embeddings[i]
            updated = self.update_alpha * old_emb + (1.0 - self.update_alpha) * new_emb
            self.active_people[pid]["embedding"] = self._normalize(updated)
            self.active_people[pid]["last_seen"] = float(timestamp)
            self.disappeared_frames[pid] = 0

        for i, face_data in enumerate(faces):
            if assigned_ids[i] is None:
                assigned_ids[i] = self._register(face_data["embedding"], timestamp)

        for pid in list(self.active_people.keys()):
            if pid not in used_people:
                self.disappeared_frames[pid] += 1
                if self.disappeared_frames[pid] > self.max_disappeared:
                    del self.active_people[pid]
                    del self.disappeared_frames[pid]

        return assigned_ids


def create_unique_color(identifier):
    """Create a unique color based on the identifier."""
    hash_val = hash(identifier)
    r = (hash_val & 0xFF0000) >> 16
    g = (hash_val & 0x00FF00) >> 8
    b = hash_val & 0x0000FF
    return (b, g, r)


def get_emotion_color(emotion):
    """Return color based on emotion type."""
    emotion_colors = {
        "angry": (0, 0, 255),      # Red
        "disgust": (0, 128, 0),    # Dark Green
        "fear": (128, 0, 128),     # Purple
        "happy": (0, 255, 255),    # Yellow
        "sad": (255, 0, 0),        # Blue
        "surprise": (0, 165, 255), # Orange
        "neutral": (128, 128, 128) # Gray
    }
    return emotion_colors.get(emotion.lower(), (255, 255, 255))


print("✅ Helper functions and FaceTracker class defined")

✅ Helper functions and FaceTracker class defined


## 6. Main Processing (GPU/CPU Auto-Detection)

In [7]:
# ========================================
# MAIN PROCESSING (GPU/CPU Auto-Detection)
# ========================================

from insightface.app import FaceAnalysis
from deepface import DeepFace
import subprocess

# Emotion smoothing settings
EMOTION_WINDOW_SECONDS = 5.0
MIN_CONFIDENCE_THRESHOLD = 30.0

update_status("PROCESSING", "Starting face detection and emotion analysis...")

# ========================================
# AUTO-DETECT GPU/CPU
# ========================================
print("🔍 Detecting hardware...")

import onnxruntime as ort
available_providers = ort.get_available_providers()

# Auto-select best available provider
if "CUDAExecutionProvider" in available_providers:
    providers_to_use = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    USE_GPU = True
    print("✅ GPU detected! Using CUDA acceleration.")
else:
    providers_to_use = ["CPUExecutionProvider"]
    USE_GPU = False
    print("ℹ️ GPU not available. Using CPU mode (slower but works).")
    print("   Tip: For faster processing, enable GPU in Colab:")
    print("   Runtime > Change runtime type > GPU")

print(f"   Available providers: {available_providers}")
print(f"   Using: {providers_to_use}")

# Initialize InsightFace for face detection
print("\n📦 Loading face detection model...")
app = FaceAnalysis(name="buffalo_s", providers=providers_to_use)
app.prepare(ctx_id=0 if USE_GPU else -1, det_size=(640, 640))
print("✅ Face detection model loaded!")

# Open input video
cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
if not cap.isOpened():
    update_status("FAILED", f"Input video not found: {INPUT_VIDEO_PATH}")
    raise FileNotFoundError(f"Video not found: {INPUT_VIDEO_PATH}")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Create video writer - use temp file for conversion
temp_video_path = OUTPUT_VIDEO_PATH.replace('.mp4', '_temp.avi')
fourcc = cv2.VideoWriter_fourcc(*"XVID")
out = cv2.VideoWriter(temp_video_path, fourcc, fps, (width, height))

print(f"\n📹 Video loaded: {total_frames} frames @ {fps:.1f} FPS")
print(f"   Resolution: {width}x{height}")
print(f"   Mode: {'GPU' if USE_GPU else 'CPU'}")
print(f"   Emotion smoothing: {EMOTION_WINDOW_SECONDS}s window")

# Initialize face tracker
tracker = FaceTracker(similarity_threshold=0.45, max_disappeared=50)

# Storage for results
all_emotion_detections = []
face_detections_for_emotion = {}
emotion_history = defaultdict(list)
stable_emotions = {}

frame_idx = 0
job_terminated = False

# ========================================
# PHASE 1: Face Detection with InsightFace
# ========================================
update_status("PROCESSING", "Phase 1/2: Detecting and tracking faces...")
print("\n🔍 Phase 1: Face Detection...")

while cap.isOpened():
    # Check for termination every 30 frames
    if frame_idx % 30 == 0 and check_termination():
        print("🛑 Job terminated by user. Stopping processing...")
        job_terminated = True
        break

    success, frame = cap.read()
    if not success:
        break

    frame_idx += 1
    timestamp = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
    ts_key = round(float(timestamp), 2)

    try:
        faces = app.get(frame)

        face_items = []
        for f in faces:
            if getattr(f, "embedding", None) is None:
                continue

            x1, y1, x2, y2 = f.bbox
            x = max(0, int(x1))
            y = max(0, int(y1))
            w = max(0, int(x2 - x1))
            h = max(0, int(y2 - y1))

            if w <= 0 or h <= 0:
                continue

            face_items.append({
                "embedding": np.asarray(f.embedding, dtype=np.float32),
                "bbox": [x, y, w, h],
            })

        assigned_ids = tracker.update(face_items, timestamp)

        face_detections_for_emotion[ts_key] = []
        for face, person_id in zip(face_items, assigned_ids):
            if person_id is None:
                continue
            face_detections_for_emotion[ts_key].append({
                "person_id": f"person_{int(person_id)}",
                "bbox": face["bbox"],
            })

        if frame_idx % 30 == 0:
            progress = int((frame_idx / total_frames) * 50)
            ids_str = ", ".join([f"person_{pid}" for pid in assigned_ids]) if assigned_ids else "none"
            print(f"[Frame {frame_idx}/{total_frames}] Faces: {len(face_items)} | IDs: [{ids_str}]")
            update_status("PROCESSING", f"Face detection: {progress}% ({frame_idx}/{total_frames} frames)", progress)

    except Exception as e:
        print(f"Error processing frame {frame_idx}: {str(e)}")
        continue

cap.release()

if job_terminated:
    out.release()
    cv2.destroyAllWindows()
    update_status("CANCELLED", "Job terminated by user")
    print("🛑 Processing stopped. Cleanup complete.")
    raise SystemExit("Job terminated by user")

# ========================================
# PHASE 2: Emotion Analysis with DeepFace
# ========================================
update_status("PROCESSING", "Phase 2/2: Analyzing emotions with DeepFace...")
print("\n😊 Phase 2: Emotion Analysis...")

cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
frame_idx = 0

while cap.isOpened():
    if frame_idx % 30 == 0 and check_termination():
        print("🛑 Job terminated by user. Stopping processing...")
        job_terminated = True
        break

    success, frame = cap.read()
    if not success:
        break

    frame_idx += 1
    current_ts = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
    ts_key = round(float(current_ts), 2)

    current_faces = face_detections_for_emotion.get(ts_key, [])

    for face in current_faces:
        pid = face["person_id"]
        x, y, w, h = face["bbox"]

        face_roi = frame[y:y+h, x:x+w]

        if face_roi.size == 0:
            continue

        try:
            result = DeepFace.analyze(
                face_roi,
                actions=["emotion"],
                enforce_detection=False,
                silent=True
            )

            if isinstance(result, list):
                result = result[0]

            emotion_scores = result["emotion"]
            emotion_history[pid].append((current_ts, emotion_scores))

            cutoff_time = current_ts - EMOTION_WINDOW_SECONDS
            emotion_history[pid] = [(ts, scores) for ts, scores in emotion_history[pid] if ts >= cutoff_time]

            accumulated_scores = defaultdict(float)
            for ts, scores in emotion_history[pid]:
                for emotion, score in scores.items():
                    accumulated_scores[emotion] += score

            dominant_emotion = max(accumulated_scores, key=accumulated_scores.get)
            confidence = emotion_scores[dominant_emotion]
            stable_emotions[pid] = (dominant_emotion, confidence)

            emotion_detection = {
                "timestamp": ts_key,
                "person_id": pid,
                "coordinates_pixels": [int(x), int(y), int(w), int(h)],
                "emotion": dominant_emotion,
                "confidence": round(float(confidence), 2),
                "all_emotions": {k: round(float(v), 2) for k, v in emotion_scores.items()}
            }
            all_emotion_detections.append(emotion_detection)

            # Draw on frame
            person_color = create_unique_color(pid)
            emotion_color = get_emotion_color(dominant_emotion)

            cv2.rectangle(frame, (x, y), (x + w, y + h), person_color, 2)

            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.5
            thickness = 2

            (text_w, text_h), _ = cv2.getTextSize(pid, font, font_scale, thickness)
            cv2.rectangle(frame, (x, y - text_h - 10), (x + text_w + 10, y), person_color, -1)
            cv2.putText(frame, pid, (x + 5, y - 5), font, font_scale, (255, 255, 255), thickness)

            label_emotion = f"{dominant_emotion} ({confidence:.0f}%)"
            (emo_w, emo_h), _ = cv2.getTextSize(label_emotion, font, font_scale, thickness)
            cv2.rectangle(frame, (x, y + h), (x + emo_w + 10, y + h + emo_h + 10), emotion_color, -1)
            cv2.putText(frame, label_emotion, (x + 5, y + h + emo_h + 5), font, font_scale, (255, 255, 255), thickness)

        except Exception as e:
            person_color = create_unique_color(pid)
            cv2.rectangle(frame, (x, y), (x + w, y + h), person_color, 2)
            cv2.putText(frame, pid, (x + 5, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, person_color, 2)

    cv2.putText(frame, f"Frame: {frame_idx}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    out.write(frame)

    if frame_idx % 30 == 0:
        progress = 50 + int((frame_idx / total_frames) * 50)
        print(f"[Frame {frame_idx}/{total_frames}] Emotions detected")
        update_status("PROCESSING", f"Emotion analysis: {progress}% ({frame_idx}/{total_frames} frames)", progress)

cap.release()
out.release()
cv2.destroyAllWindows()

if job_terminated:
    update_status("CANCELLED", "Job terminated by user during emotion analysis")
    print("🛑 Processing stopped. Cleanup complete.")
    raise SystemExit("Job terminated by user")

# ========================================
# CONVERT TO H.264 MP4 (Browser Compatible)
# ========================================
print("\n🔄 Converting video to H.264 MP4 for browser compatibility...")

if os.path.exists(temp_video_path):
    # Try ffmpeg conversion
    ffmpeg_cmd = f'ffmpeg -i "{temp_video_path}" -c:v libx264 -preset fast -crf 23 -pix_fmt yuv420p -y "{OUTPUT_VIDEO_PATH}" -loglevel error'
    result = subprocess.run(ffmpeg_cmd, shell=True, capture_output=True, text=True)

    if result.returncode == 0 and os.path.exists(OUTPUT_VIDEO_PATH):
        os.remove(temp_video_path)
        file_size = os.path.getsize(OUTPUT_VIDEO_PATH) / (1024 * 1024)
        print(f"✅ Video converted to H.264 MP4!")
        print(f"   Size: {file_size:.2f} MB")
    else:
        print(f"⚠️ ffmpeg conversion failed, using fallback...")
        import shutil
        shutil.move(temp_video_path, OUTPUT_VIDEO_PATH)
        print("   Video saved (may not play in all browsers)")
else:
    print(f"⚠️ Temp video not found")

print(f"\n✅ Processing complete!")
print(f"  - Total emotion detections: {len(all_emotion_detections)}")
print(f"  - Output video: {OUTPUT_VIDEO_PATH}")
print(f"  - Output JSON: {OUTPUT_JSON_PATH}")

📝 Status: PROCESSING - Starting face detection and emotion analysis...
🔍 Detecting hardware...
✅ GPU detected! Using CUDA acceleration.
   Available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
   Using: ['CUDAExecutionProvider', 'CPUExecutionProvider']

📦 Loading face detection model...
download_path: /root/.insightface/models/buffalo_s


100%|██████████| 124617/124617 [00:01<00:00, 81563.31KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_s/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_s/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_s/det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_s/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_s/w600k_mbf.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
✅ Face dete

Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5


26-01-02 20:42:08 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


100%|██████████| 5.98M/5.98M [00:00<00:00, 77.3MB/s]



🔄 Converting video to H.264 MP4 for browser compatibility...
✅ Video converted to H.264 MP4!
   Size: 0.20 MB

✅ Processing complete!
  - Total emotion detections: 216
  - Output video: /content/drive/MyDrive/ai_emotion_project/output_videos/186cc8fb-870c-418b-ba3a-c8042ca83a82.mp4
  - Output JSON: /content/drive/MyDrive/ai_emotion_project/output_json/186cc8fb-870c-418b-ba3a-c8042ca83a82.json


## 7. Calculate Emotion Summary

In [8]:
# ========================================
# CALCULATE EMOTION SUMMARY
# ========================================

# Aggregate emotions per person
emotion_summary = {}

for detection in all_emotion_detections:
    pid = detection["person_id"]
    emotion = detection["emotion"]

    if pid not in emotion_summary:
        emotion_summary[pid] = {
            "frame_count": 0,
            "emotions": defaultdict(int)
        }

    emotion_summary[pid]["frame_count"] += 1
    emotion_summary[pid]["emotions"][emotion] += 1

# Convert defaultdict to regular dict for JSON serialization
for pid in emotion_summary:
    emotion_summary[pid]["emotions"] = dict(emotion_summary[pid]["emotions"])

print(f"✅ Emotion summary calculated for {len(emotion_summary)} person(s)")

✅ Emotion summary calculated for 9 person(s)


## 8. Save Results to Google Drive

In [9]:
# Save emotion summary to JSON (format expected by web app)
output_data = {
    "summary_by_person": emotion_summary,
    "total_frames": frame_idx,
    "total_detections": len(all_emotion_detections)
}

with open(OUTPUT_JSON_PATH, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"\n✅ Results saved:")
print(f"Output video: {OUTPUT_VIDEO_PATH}")
print(f"Output JSON: {OUTPUT_JSON_PATH}")

# Display emotion summary
print("\n📊 Emotion Summary:")
for person_id, data in emotion_summary.items():
    print(f"\n{person_id.upper()}:")
    print(f"  Total frames: {data['frame_count']}")
    print(f"  Emotions:")
    for emotion, count in sorted(data['emotions'].items(), key=lambda x: x[1], reverse=True):
        percentage = (count / data['frame_count']) * 100
        print(f"    {emotion}: {count} frames ({percentage:.1f}%)")

# Final status update
update_status("DONE", f"Processing complete! Detected {len(emotion_summary)} person(s) across {frame_idx} frames")

print("\n🎉 ALL DONE! Check your web application for results.")


✅ Results saved:
Output video: /content/drive/MyDrive/ai_emotion_project/output_videos/186cc8fb-870c-418b-ba3a-c8042ca83a82.mp4
Output JSON: /content/drive/MyDrive/ai_emotion_project/output_json/186cc8fb-870c-418b-ba3a-c8042ca83a82.json

📊 Emotion Summary:

PERSON_1:
  Total frames: 24
  Emotions:
    happy: 24 frames (100.0%)

PERSON_2:
  Total frames: 24
  Emotions:
    happy: 24 frames (100.0%)

PERSON_3:
  Total frames: 24
  Emotions:
    sad: 23 frames (95.8%)
    fear: 1 frames (4.2%)

PERSON_4:
  Total frames: 24
  Emotions:
    neutral: 24 frames (100.0%)

PERSON_5:
  Total frames: 24
  Emotions:
    neutral: 24 frames (100.0%)

PERSON_6:
  Total frames: 24
  Emotions:
    happy: 24 frames (100.0%)

PERSON_7:
  Total frames: 24
  Emotions:
    neutral: 23 frames (95.8%)
    happy: 1 frames (4.2%)

PERSON_8:
  Total frames: 24
  Emotions:
    happy: 24 frames (100.0%)

PERSON_9:
  Total frames: 24
  Emotions:
    fear: 24 frames (100.0%)
📝 Status: DONE - Processing complete! De

## 9. Verify Output Files

In [10]:
# Verify all output files exist
print("Verifying output files...\n")

files_to_check = [
    (OUTPUT_VIDEO_PATH, "Output video"),
    (OUTPUT_JSON_PATH, "Emotion JSON"),
    (STATUS_FILE_PATH, "Status file")
]

all_exist = True
for file_path, description in files_to_check:
    exists = os.path.exists(file_path)
    status = "✅" if exists else "❌"
    size = os.path.getsize(file_path) if exists else 0
    print(f"{status} {description}: {file_path}")
    if exists:
        print(f"   Size: {size / (1024*1024):.2f} MB")
    all_exist = all_exist and exists

if all_exist:
    print("\n✅ All files created successfully!")
    print("\n👉 Go to your web application to view results.")
else:
    print("\n❌ Some files are missing. Check for errors above.")

Verifying output files...

✅ Output video: /content/drive/MyDrive/ai_emotion_project/output_videos/186cc8fb-870c-418b-ba3a-c8042ca83a82.mp4
   Size: 0.20 MB
✅ Emotion JSON: /content/drive/MyDrive/ai_emotion_project/output_json/186cc8fb-870c-418b-ba3a-c8042ca83a82.json
   Size: 0.00 MB
✅ Status file: /content/drive/MyDrive/ai_emotion_project/status/186cc8fb-870c-418b-ba3a-c8042ca83a82.json
   Size: 0.00 MB

✅ All files created successfully!

👉 Go to your web application to view results.


---

## 📝 Notes

### Supported Modes:
- **🌐 Google Colab** (recommended): Set `USE_LOCAL_MACHINE = False`
- **💻 Local Machine**: Set `USE_LOCAL_MACHINE = True`

### Hardware Support:
- **GPU (CUDA)**: Auto-detected, used if available (faster)
- **CPU**: Automatic fallback if GPU not available (slower but works)

### What This Notebook Does:
- ✅ Auto-detects GPU/CPU and uses the best available
- ✅ Reads video from Google Drive (Colab) or local folder
- ✅ Detects faces with InsightFace
- ✅ Detects emotions with DeepFace
- ✅ Annotates video frames with person IDs and emotions
- ✅ Converts output to H.264 MP4 (browser compatible)
- ✅ Writes emotion JSON for web app
- ✅ Updates status for live monitoring
- ✅ Supports job termination from web app

### Workflow:
1. Upload video via web app → saves to Google Drive
2. Copy the job_id from the web app
3. Set `JOB_ID` in configuration cell
4. Set `USE_LOCAL_MACHINE` based on your environment
5. **Run All** (Ctrl+F9 or Runtime > Run all)
6. Web app displays results when processing completes

### Tips:
- **For Colab**: Use GPU runtime for faster processing
- **For Local**: Make sure ffmpeg is installed for video conversion
- **Job ID**: Must match exactly (copy-paste recommended)
- **Termination**: Click "Terminate" in web app to stop processing

### Troubleshooting:
- **"Video not found"**: Check JOB_ID and that video was uploaded
- **"GPU not available"**: Use CPU mode (slower) or enable GPU in Colab
- **Video won't play**: ffmpeg conversion may have failed, check output

---